In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import ops

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
ROOT_DIR = r"E:\WISDM_ar_v1.1"

ABLATION_ROOT = os.path.join(ROOT_DIR, "ABLATION_STUDY")

WISDM_OUT = os.path.join(ABLATION_ROOT, "WISDM")

DIRS = {
    "models": os.path.join(WISDM_OUT, "models"),
    "reports": os.path.join(WISDM_OUT, "reports"),
    "plots": os.path.join(WISDM_OUT, "plots"),
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("Folders Created Successfully")

In [ ]:
WISDM_FILE = os.path.join(
    ROOT_DIR,
    "WISDM_ar_v1.1_raw.txt"
)

LABELS = [
    "Downstairs",
    "Jogging",
    "Sitting",
    "Standing",
    "Upstairs",
    "Walking"
]

label2idx = {lab: i for i, lab in enumerate(LABELS)}

rows = []

with open(WISDM_FILE, "r") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        if line.endswith(";"):
            line = line[:-1]

        parts = line.split(",")

        if len(parts) != 6:
            continue

        user, act, ts, x, y, z = parts

        if act not in label2idx:
            continue

        rows.append([user, act, ts, x, y, z])

df = pd.DataFrame(
    rows,
    columns=[
        "user",
        "activity",
        "timestamp",
        "x",
        "y",
        "z"
    ]
)

for c in ["timestamp", "x", "y", "z"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.dropna(inplace=True)

df = df[df["timestamp"] > 0]

df.sort_values(["user", "timestamp"], inplace=True)

df.reset_index(drop=True, inplace=True)

print(df.shape)

df.head()

In [ ]:
FEATURES = ["x", "y", "z"]

WIN = 80
STEP = 40

Xw = []
yw = []

for user_id, sub in df.groupby("user", sort=False):

    X = sub[FEATURES].values.astype(np.float32)

    y = sub["activity"].map(label2idx).values.astype(np.int32)

    for i in range(0, len(X) - WIN + 1, STEP):

        xs = X[i:i+WIN]
        ys = y[i:i+WIN]

        label = np.bincount(ys).argmax()

        Xw.append(xs)
        yw.append(label)

Xw = np.array(Xw, dtype=np.float32)
yw = np.array(yw, dtype=np.int32)

print("Windows Shape:", Xw.shape)
print("Labels Shape:", yw.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    Xw,
    yw,
    test_size=0.2,
    stratify=yw,
    random_state=SEED
)

scaler = StandardScaler()

X_train_s = scaler.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_s = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

X_train_s = X_train_s.astype(np.float32)
X_test_s = X_test_s.astype(np.float32)

print("Train:", X_train_s.shape)
print("Test:", X_test_s.shape)

In [ ]:
def encoder_block(
    x,
    d_model,
    num_heads,
    mlp_dim,
    dropout
):

    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + attn)

    ff = layers.Dense(
        mlp_dim,
        activation="gelu"
    )(x)

    ff = layers.Dropout(dropout)(ff)

    ff = layers.Dense(d_model)(ff)

    ff = layers.Dropout(dropout)(ff)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + ff)

    return x

In [ ]:
def build_transformer(
    input_shape,
    num_classes,
    patch_len,
    depth
):

    d_model = 64
    num_heads = 4
    mlp_dim = 128
    dropout = 0.15

    inp = keras.Input(shape=input_shape)

    x = layers.Conv1D(
        filters=d_model,
        kernel_size=patch_len,
        strides=patch_len,
        padding="valid"
    )(inp)

    seq_len = ops.shape(x)[1]

    pos = ops.arange(
        start=0,
        stop=seq_len,
        step=1
    )

    pos_emb = layers.Embedding(
        input_dim=2048,
        output_dim=d_model
    )(pos)

    pos_emb = ops.expand_dims(pos_emb, axis=0)

    x = x + pos_emb

    x = layers.Dropout(dropout)(x)

    for _ in range(depth):

        x = encoder_block(
            x,
            d_model=d_model,
            num_heads=num_heads,
            mlp_dim=mlp_dim,
            dropout=dropout
        )

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dropout(dropout)(x)

    out = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = keras.Model(inp, out)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
PATCH_LIST = [1, 2, 4, 8, 16, 32]

DEPTH_LIST = [1, 2, 3, 4]

configs = []

for patch_len in PATCH_LIST:

    for depth in DEPTH_LIST:

        name = f"Patch{patch_len}_Depth{depth}"

        if patch_len == 1:
            name = f"NoPatch_Depth{depth}"

        configs.append({
            "name": name,
            "patch_len": patch_len,
            "depth": depth
        })

print("Total Experiments:", len(configs))

configs[:5]

In [ ]:
def evaluate_model(model):

    probs = model.predict(
        X_test_s,
        verbose=0
    )

    preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_test, preds)

    f1 = f1_score(
        y_test,
        preds,
        average="macro"
    )

    return acc, f1


def measure_latency(model):

    sample = X_test_s[:1]

    # warmup
    for _ in range(10):
        model.predict(sample, verbose=0)

    start = time.time()

    for _ in range(100):
        model.predict(sample, verbose=0)

    end = time.time()

    latency = ((end - start) / 100) * 1000

    return latency

In [ ]:
results = []

for cfg in configs:

    print("\n" + "="*80)
    print("Training:", cfg["name"])
    print("="*80)

    tf.keras.backend.clear_session()

    model = build_transformer(
        input_shape=X_train_s.shape[1:],
        num_classes=6,
        patch_len=cfg["patch_len"],
        depth=cfg["depth"]
    )

    callbacks = [

        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

    history = model.fit(
        X_train_s,
        y_train,
        validation_split=0.1,
        epochs=20,
        batch_size=128,
        callbacks=callbacks,
        verbose=1
    )

    acc, f1 = evaluate_model(model)

    latency = measure_latency(model)

    # Save model
    model_path = os.path.join(
        DIRS["models"],
        cfg["name"] + ".keras"
    )

    model.save(model_path)

    size_kb = os.path.getsize(model_path) / 1024

    params = model.count_params()

    results.append({

        "Configuration": cfg["name"],

        "Patch Length": cfg["patch_len"],

        "Encoder Layers": cfg["depth"],

        "Accuracy (%)": round(acc * 100, 2),

        "Macro F1 (%)": round(f1 * 100, 2),

        "Latency (ms)": round(latency, 2),

        "Model Size (KB)": round(size_kb, 2),

        "Parameters": params
    })

    temp_df = pd.DataFrame(results)

    running_csv = os.path.join(
        DIRS["reports"],
        "WISDM_RUNNING_RESULTS.csv"
    )

    temp_df.to_csv(running_csv, index=False)

    print(temp_df.tail(1))

In [ ]:
results_df = pd.DataFrame(results)

final_csv = os.path.join(
    DIRS["reports"],
    "WISDM_FINAL_RESULTS.csv"
)

results_df.to_csv(final_csv, index=False)

print("Saved:", final_csv)

results_df

In [ ]:
pivot_acc = results_df.pivot(
    index="Patch Length",
    columns="Encoder Layers",
    values="Accuracy (%)"
)

plt.figure(figsize=(8,5))

plt.imshow(pivot_acc, aspect="auto")

plt.xticks(
    range(len(pivot_acc.columns)),
    pivot_acc.columns
)

plt.yticks(
    range(len(pivot_acc.index)),
    pivot_acc.index
)

for i in range(len(pivot_acc.index)):

    for j in range(len(pivot_acc.columns)):

        plt.text(
            j,
            i,
            f"{pivot_acc.iloc[i,j]:.2f}",
            ha="center",
            va="center"
        )

plt.xlabel("Encoder Layers")

plt.ylabel("Patch Length")

plt.title("WISDM Accuracy Heatmap")

plt.colorbar(label="Accuracy (%)")

plot_path = os.path.join(
    DIRS["plots"],
    "WISDM_ACCURACY_HEATMAP.png"
)

plt.savefig(
    plot_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", plot_path)

In [ ]:
pivot_f1 = results_df.pivot(
    index="Patch Length",
    columns="Encoder Layers",
    values="Macro F1 (%)"
)

plt.figure(figsize=(8,5))

plt.imshow(pivot_f1, aspect="auto")

plt.xticks(
    range(len(pivot_f1.columns)),
    pivot_f1.columns
)

plt.yticks(
    range(len(pivot_f1.index)),
    pivot_f1.index
)

for i in range(len(pivot_f1.index)):

    for j in range(len(pivot_f1.columns)):

        plt.text(
            j,
            i,
            f"{pivot_f1.iloc[i,j]:.2f}",
            ha="center",
            va="center"
        )

plt.xlabel("Encoder Layers")

plt.ylabel("Patch Length")

plt.title("WISDM Macro F1 Heatmap")

plt.colorbar(label="Macro F1 (%)")

plot_path = os.path.join(
    DIRS["plots"],
    "WISDM_MACRO_F1_HEATMAP.png"
)

plt.savefig(
    plot_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", plot_path)

In [ ]:
pivot_latency = results_df.pivot(
    index="Patch Length",
    columns="Encoder Layers",
    values="Latency (ms)"
)

plt.figure(figsize=(8,5))

plt.imshow(pivot_latency, aspect="auto")

plt.xticks(
    range(len(pivot_latency.columns)),
    pivot_latency.columns
)

plt.yticks(
    range(len(pivot_latency.index)),
    pivot_latency.index
)

for i in range(len(pivot_latency.index)):

    for j in range(len(pivot_latency.columns)):

        plt.text(
            j,
            i,
            f"{pivot_latency.iloc[i,j]:.2f}",
            ha="center",
            va="center"
        )

plt.xlabel("Encoder Layers")

plt.ylabel("Patch Length")

plt.title("WISDM Latency Heatmap")

plt.colorbar(label="Latency (ms)")

plot_path = os.path.join(
    DIRS["plots"],
    "WISDM_LATENCY_HEATMAP.png"
)

plt.savefig(
    plot_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", plot_path)

In [ ]:
# ============================================================
# CELL 15 — Load Best Balanced WISDM Model: Patch8_Depth2
# ============================================================

BEST_CONFIG = "Patch8_Depth2"

best_model_path = os.path.join(
    DIRS["models"],
    BEST_CONFIG + ".keras"
)

best_model = keras.models.load_model(best_model_path)

print("Loaded model:", best_model_path)
best_model.summary()

In [ ]:
# ============================================================
# CELL 16 — TFLite Conversion Functions
# ============================================================

import tempfile

def convert_to_tflite(model, quant_type="fp32", representative_data=None):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quant_type == "fp32":
        pass

    elif quant_type == "fp16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == "int8_dynamic":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    elif quant_type == "int8_full":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        def representative_dataset():
            for i in range(min(300, len(representative_data))):
                yield [representative_data[i:i+1].astype(np.float32)]

        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    return converter.convert()

In [ ]:
# ============================================================
# CELL 17 — TFLite Evaluation + Latency
# ============================================================

def evaluate_tflite_model(tflite_model, X_test, y_test):
    with tempfile.NamedTemporaryFile(suffix=".tflite", delete=False) as f:
        f.write(tflite_model)
        temp_model_path = f.name

    interpreter = tf.lite.Interpreter(model_path=temp_model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    preds = []

    for i in range(len(X_test)):
        x = X_test[i:i+1].astype(np.float32)

        if input_details[0]["dtype"] == np.int8:
            scale, zero_point = input_details[0]["quantization"]
            x = x / scale + zero_point
            x = np.clip(x, -128, 127).astype(np.int8)

        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

        out = interpreter.get_tensor(output_details[0]["index"])

        if output_details[0]["dtype"] == np.int8:
            scale, zero_point = output_details[0]["quantization"]
            out = scale * (out.astype(np.float32) - zero_point)

        preds.append(np.argmax(out))

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    return acc, f1


def measure_tflite_latency(tflite_model, X_test, warmup=30, runs=300):
    with tempfile.NamedTemporaryFile(suffix=".tflite", delete=False) as f:
        f.write(tflite_model)
        temp_model_path = f.name

    interpreter = tf.lite.Interpreter(model_path=temp_model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()

    x = X_test[:1].astype(np.float32)

    if input_details[0]["dtype"] == np.int8:
        scale, zero_point = input_details[0]["quantization"]
        x = x / scale + zero_point
        x = np.clip(x, -128, 127).astype(np.int8)

    for _ in range(warmup):
        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

    start = time.time()

    for _ in range(runs):
        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()

    end = time.time()

    latency = ((end - start) / runs) * 1000

    return latency

In [ ]:
# ============================================================
# CELL 18 — Quantization Study for Patch8_Depth2
# ============================================================

TFLITE_DIR = os.path.join(WISDM_OUT, "tflite")
os.makedirs(TFLITE_DIR, exist_ok=True)

quant_results = []

# Add Keras FP32 baseline for same model
keras_acc, keras_f1 = evaluate_model(best_model)
keras_latency = measure_latency(best_model)

keras_size_kb = os.path.getsize(best_model_path) / 1024

quant_results.append({
    "Configuration": BEST_CONFIG,
    "Variant": "KERAS_FP32",
    "Patch Length": 8,
    "Encoder Layers": 2,
    "Accuracy (%)": round(keras_acc * 100, 2),
    "Macro F1 (%)": round(keras_f1 * 100, 2),
    "Latency (ms)": round(keras_latency, 2),
    "Model Size (KB)": round(keras_size_kb, 2),
    "Compression Ratio": round(keras_size_kb / keras_size_kb, 2)
})

for qtype in ["fp32", "fp16", "int8_dynamic", "int8_full"]:

    print("\nConverting:", qtype)

    tflite_model = convert_to_tflite(
        best_model,
        quant_type=qtype,
        representative_data=X_train_s
    )

    tflite_path = os.path.join(
        TFLITE_DIR,
        f"{BEST_CONFIG}_{qtype}.tflite"
    )

    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    acc, f1 = evaluate_tflite_model(
        tflite_model,
        X_test_s,
        y_test
    )

    latency = measure_tflite_latency(
        tflite_model,
        X_test_s
    )

    size_kb = os.path.getsize(tflite_path) / 1024

    quant_results.append({
        "Configuration": BEST_CONFIG,
        "Variant": f"TFLITE_{qtype.upper()}",
        "Patch Length": 8,
        "Encoder Layers": 2,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro F1 (%)": round(f1 * 100, 2),
        "Latency (ms)": round(latency, 2),
        "Model Size (KB)": round(size_kb, 2),
        "Compression Ratio": round(keras_size_kb / size_kb, 2)
    })

    print(quant_results[-1])

In [ ]:
# ============================================================
# CELL 19 — Save Quantization Results
# ============================================================

quant_df = pd.DataFrame(quant_results)

quant_csv = os.path.join(
    DIRS["reports"],
    "WISDM_Patch8_Depth2_QUANTIZATION_RESULTS.csv"
)

quant_df.to_csv(quant_csv, index=False)

print("Saved:", quant_csv)

quant_df

In [ ]:
# ============================================================
# CELL 20 — Merge Grid Ablation + Quantization Study
# ============================================================

grid_df = pd.read_csv(
    os.path.join(DIRS["reports"], "WISDM_FINAL_RESULTS.csv")
)

grid_df["Variant"] = "KERAS_FP32"

grid_df = grid_df[
    [
        "Configuration",
        "Variant",
        "Patch Length",
        "Encoder Layers",
        "Accuracy (%)",
        "Macro F1 (%)",
        "Latency (ms)",
        "Model Size (KB)",
        "Parameters"
    ]
]

quant_df_for_merge = quant_df.copy()
quant_df_for_merge["Parameters"] = ""

# avoid duplicate KERAS_FP32 row for Patch8_Depth2
quant_df_for_merge = quant_df_for_merge[
    quant_df_for_merge["Variant"] != "KERAS_FP32"
]

final_ablation_df = pd.concat(
    [grid_df, quant_df_for_merge],
    ignore_index=True
)

final_csv = os.path.join(
    DIRS["reports"],
    "WISDM_FINAL_GRID_PLUS_QUANTIZATION_ABLATION.csv"
)

final_ablation_df.to_csv(final_csv, index=False)

print("Saved final table:", final_csv)

final_ablation_df